# Repository working directory

In [ ]:
# Execute this notebook from the repository root.
# Set the current working directory to the root before starting the R kernel.
print(getwd())


# Load project configuration and required R packages

In [ ]:
source("R/00_config.R")
library(repr)
library(tidyverse)
# library(ggpubr) library(rstatix) library(patchwork) library(nortest)
# library(purrr)
library(tidyr)
# library(readr) library(lme4) library(broom.mixed) library(lmerTest)
# Analysis step: see following code.
# library(MASS)
library(dplyr)
# library(purrr)
library(ggplot2)
# library(broom) library(randomForest)
library(missForest)
library(xgboost)
library(caret)
library(Mime1)
library(multiROC)
library(DMwR)
library(shapviz)
library(boot)
library(parallel)
library(doParallel)


In [ ]:
set.seed(5201314)


# Load and organize the shared resampling data

In [ ]:
# Load the shared resampled training and held-out testing partitions.
train_val_list <- read_rds(file.path(INTERMEDIATE_DIR, "resampling_data.rds"))


In [ ]:
print(str(train_val_list$balanced_training))


In [ ]:
train_df <- train_val_list$balanced_training
test_df <- train_val_list$balanced_validation


In [ ]:
irrelevant_features <- c("Clinical.diagnosis", "ImageID", "ScanGroup")
features <- setdiff(names(train_val_list$balanced_training), irrelevant_features)


In [ ]:
print(features)


# Classifier hyperparameter configurations

In [ ]:
weight_power_range <- seq(0,2,by=0.5)
Grid <- list(
    svmRadial = expand.grid(sigma = c(0.01, 0.02, 0.03, 0.04, 0.05), C = c(0.1, 1, 10, 15)),
    glmnet = expand.grid(alpha = seq(0,1,by=0.2), lambda = c(0.01, 0.1, 1, 10)),
    multinom = expand.grid(decay = c(1, 10, 15, 20)),
    pda = expand.grid(lambda = c(50,100,150)),
)
ranger_param <- list(splitrule = c("gini"), min.node.size = c(7, 10, 15))


# Classification training and evaluation configuration

## Original metric calculation helper (reference implementation)

In [ ]:
multi_auc_summary_custom <- function(data, lev = NULL, model = NULL) {
  
  if (is.null(lev)) {
    lev <- levels(data$obs)
  }
  
  cm <- confusionMatrix(
    factor(data$pred, levels = lev),
    factor(data$obs, levels = lev)
  )
  
  tab <- cm$table
  TP <- sum(diag(tab))
  FP <- sum(colSums(tab) - diag(tab))
  FN <- sum(rowSums(tab) - diag(tab))
  
  y_true <- as.data.frame(
    model.matrix(~ obs - 1, data = data)
  )
  
  names(y_true) <- gsub("^obs", "", names(y_true))
  names(y_true) <- paste0(names(y_true), "_true")
  
  pred_prob <- data[, lev, drop = FALSE]
  names(pred_prob) <- paste0(names(pred_prob), "_pred_xgb")
  
  roc_res <- multi_roc(dplyr::bind_cols(y_true, pred_prob))
  
  c(
    accuracy = unname(cm$overall["Accuracy"]),
    kappa = unname(cm$overall["Kappa"]),
    balance_accuracy = mean(cm$byClass[, "Balanced Accuracy"], na.rm = TRUE),
    micro_f1 = 2 * TP / (2 * TP + FN + FP),
    macro_f1 = mean(cm$byClass[, "F1"], na.rm = TRUE),
    micro_auc = roc_res$AUC$xgb$micro,
    macro_auc = roc_res$AUC$xgb$macro
  )
}


## Metric calculation helper used for sensitivity evaluation

In [ ]:
multi_auc_summary <- function(data, lev = NULL, model = NULL) {
  
  if (is.null(lev)) {
    lev <- levels(data$obs)
  }
  
  data$obs  <- factor(data$obs, levels = lev)
  data$pred <- factor(data$pred, levels = lev)
  
  cm <- caret::confusionMatrix(data$pred, data$obs)
  
  acc <- unname(cm$overall["Accuracy"])
  kappa <- unname(cm$overall["Kappa"])
  
  by_class <- cm$byClass
  
  if (is.matrix(by_class)) {
    balanced_accuracy <- mean(by_class[, "Balanced Accuracy"], na.rm = TRUE)
    macro_f1 <- mean(by_class[, "F1"], na.rm = TRUE)
  } else {
    balanced_accuracy <- unname(by_class["Balanced Accuracy"])
    macro_f1 <- unname(by_class["F1"])
  }
  
  tab <- cm$table
  TP <- sum(diag(tab))
  FP <- sum(colSums(tab) - diag(tab))
  FN <- sum(rowSums(tab) - diag(tab))
  
  micro_f1 <- ifelse(
    (2 * TP + FN + FP) == 0,
    NA_real_,
    2 * TP / (2 * TP + FN + FP)
  )
    
  safe_auc <- function(response, predictor) {
  
  response <- as.numeric(response)
  predictor <- suppressWarnings(as.numeric(as.character(predictor)))
  
  ok <- !is.na(response) & !is.na(predictor)
  response <- response[ok]
  predictor <- predictor[ok]
  
  if (length(response) < 2) return(NA_real_)
  if (length(unique(response)) < 2) return(NA_real_)
  if (length(unique(predictor)) < 2) return(NA_real_)
  
  as.numeric(
    pROC::auc(
      pROC::roc(
        response = response,
        predictor = predictor,
        quiet = TRUE,
        direction = "<"
      )
    )
  )
}
  
  auc_each_class <- sapply(lev, function(cls) {
    
    if (!cls %in% colnames(data)) {
      return(NA_real_)
    }
    
    truth_bin <- as.numeric(data$obs == cls)
    prob_vec <- data[[cls]]
    
    safe_auc(truth_bin, prob_vec)
  })
  
  macro_auc <- mean(auc_each_class, na.rm = TRUE)
  
  if (is.nan(macro_auc)) {
    macro_auc <- NA_real_
  }
  
  y_true_micro <- unlist(lapply(lev, function(cls) {
    as.numeric(data$obs == cls)
  }))
  
  y_score_micro <- unlist(lapply(lev, function(cls) {
    as.numeric(as.character(data[[cls]]))
  }))
  
  micro_auc <- safe_auc(y_true_micro, y_score_micro)
  
  c(
    accuracy = acc,
    kappa = kappa,
    balance_accuracy = balanced_accuracy,
    micro_f1 = micro_f1,
    macro_f1 = macro_f1,
    macro_auc = macro_auc,
    micro_auc = micro_auc
  )
}


## Configure caret training controls

### Five-fold cross-validation on the training partition

In [ ]:
ff_ctrl <- list(
  method = "repeatedcv",
  number = 5,
  repeats = 20,
  classProbs = TRUE,
  summaryFunction = multi_auc_summary,
  sampling='up',
  savePredictions = "final",
  allowParallel = TRUE,
  verboseIter = TRUE
)


### Use the predefined train/validation split

In [ ]:
cv_ctrl <- list(
  method = "cv",
  number = 1,
  classProbs = TRUE,
  summaryFunction = multi_auc_summary,
  sampling='up',
  savePredictions = "final",
  allowParallel = TRUE,
  verboseIter = TRUE
)


In [ ]:
eval_ctrl <- list(
  method = "none",
  classProbs = TRUE,
  summaryFunction = multi_auc_summary,
  sampling='up',
  savePredictions = "final",
  allowParallel = TRUE,
  verboseIter = TRUE
)


In [ ]:
methods <- setdiff(names(Grid), c('xgbTree'))
# methods <- c('kknn')
print(methods)
models <- list()
model_results <- list()
test_metrics <- list()
test_predictions <- list()


In [ ]:
# methods <- c('kknn', 'glmnet')


# Calculate confidence intervals from predicted test-set labels

In [ ]:
# Analysis step: see following code.
make_test_prediction_data <- function(fit, x_test_model, y_test) {
  
  lev <- levels(y_test)
  
  pred_class <- predict(
    fit,
    newdata = x_test_model,
    type = "raw"
  )
  
  pred_prob <- predict(
    fit,
    newdata = x_test_model,
    type = "prob"
  )
  
  pred_class <- factor(pred_class, levels = lev)
  y_test_use <- factor(y_test, levels = lev)
  
  if (!all(lev %in% colnames(pred_prob))) {
    stop(
      "Probability columns do not match outcome levels. Missing: ",
      paste(setdiff(lev, colnames(pred_prob)), collapse = ", ")
    )
  }
  
  pred_prob <- pred_prob[, lev, drop = FALSE]
  pred_prob[] <- lapply(pred_prob, function(z) {
    as.numeric(as.character(z))
  })
  
  test_data <- data.frame(
    obs = y_test_use,
    pred = pred_class,
    pred_prob,
    check.names = FALSE
  )
  
  return(test_data)
}


In [ ]:
evaluate_with_bootstrap <- function(
    test_data,
    summary_fun,
    lev,
    R = 1000,
    conf = 0.95,
    seed = 2026
) {
  
# Analysis step: see following code.
  point_est <- summary_fun(
    data = test_data,
    lev = lev,
    model = NULL
  )
  
  metric_names <- names(point_est)
  
# Load the shared resampled training and held-out testing partitions.
  boot_stat <- function(input, indices) {
    
    boot_data <- input[indices, , drop = FALSE]
    
    boot_data$obs <- factor(boot_data$obs, levels = lev)
    boot_data$pred <- factor(boot_data$pred, levels = lev)
    
    res <- summary_fun(
      data = boot_data,
      lev = lev,
      model = NULL
    )
    
# Analysis step: see following code.
    res <- res[metric_names]
    
    return(res)
  }
  
  set.seed(seed)
  
  boot_res <- boot::boot(
    data = test_data,
    statistic = boot_stat,
    R = R,
    strata = test_data$obs
  )
  
# Analysis step: see following code.
  alpha <- 1 - conf
  
  ci_lower <- apply(
    boot_res$t,
    2,
    function(z) {
      stats::quantile(z, probs = alpha / 2, na.rm = TRUE, names = FALSE)
    }
  )
  
  ci_upper <- apply(
    boot_res$t,
    2,
    function(z) {
      stats::quantile(z, probs = 1 - alpha / 2, na.rm = TRUE, names = FALSE)
    }
  )
  
  out <- data.frame(
    metric = metric_names,
    estimate = as.numeric(point_est),
    ci_lower = as.numeric(ci_lower),
    ci_upper = as.numeric(ci_upper),
    row.names = NULL
  )
  
  return(list(
    summary = out,
    boot = boot_res
  ))
}


# Manual Group

## Extract planning-group training and held-out testing data

In [ ]:
group_type <- "Manual"
x_train <- train_val_list$training %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_train <- train_val_list$training %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
x_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
ranger_param$mtry <- c(floor(sqrt(ncol(x_train)-2)/2), 
                                  floor(sqrt(ncol(x_train)-2)),
                                  floor((ncol(x_train)-2)/3), 10)
Grid$ranger <- expand.grid(ranger_param)


In [ ]:
ordered_factor_cols <- names(x_train)[sapply(x_train, is.ordered)]
# print(ordered_factor_cols)

x_train_ord <- x_train
x_test_ord <- x_test

# Analysis step: see following code.
for (v in ordered_factor_cols) {
  x_train_ord[[v]] <- as.numeric(x_train_ord[[v]])
  x_test_ord[[v]] <- as.numeric(x_test_ord[[v]])
}

# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_train_ord,
  fullRank = TRUE
)

x_train_model <- predict(dummy_obj, newdata = x_train_ord) |> as.data.frame()
x_train_model$Clinical.diagnosis <- y_train
# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_test_ord,
  fullRank = TRUE
)

x_test_model <- predict(dummy_obj, newdata = x_test_ord) |> as.data.frame()

# print(str(x_train_model))


In [ ]:
# Analysis step: see following code.
x_test_1 <- x_test_model %>% mutate(Clinical.diagnosis = y_test)
idx_train <- seq_len(nrow(x_train_model))
idx_valid <- (nrow(x_train_model) + 1):(nrow(x_train_model)+nrow(x_test_1))
x_train_model <- x_train_model %>% bind_rows(x_test_1)
rownames(x_train_model) <- seq_len(nrow(x_train_model))


## Tune classifier hyperparameters

In [ ]:
# Analysis step: see following code.
# ctrl <- ff_ctrl


cv_ctrl$sampling <- "smote"
cv_ctrl$index <- list(Holdout = idx_train)
cv_ctrl$indexOut <- list(Holdout = idx_valid)
ctrl <- cv_ctrl  # Train using the predefined partition

# Analysis step: see following code.
cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuning",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
  
  common_args <- list(
    form = as.formula("Clinical.diagnosis ~ ."),
    data = x_train_model,
#     y = y_train,
    method = m,
    tuneGrid = Grid[[m]],
    trControl = do.call(trainControl, ctrl),
    metric = "macro_auc",
    maximize = TRUE
  )
  
  if (m == "xgbTree") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          eval_metric = "mlogloss",
          verbose = 0
        )
      )
    )
    
  } else if (m == "multinom") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale"),
          trace = FALSE
        )
      )
    )
    
  } else if (m %in% c("pda", "svmRadial")) {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale")
        )
      )
    )
    
  } else {
    
    fit <- do.call(train, common_args)
  }
  
  models[[paste0(group_type,'/',m)]] <- fit
  
  model_results[[paste0(group_type,'/',m)]] <- fit$results %>%
    mutate(
      model = m,
      group = group_type
    )
  
  message(
    paste(
      "Finished", m,
      "| selected tuning parameters:",
      paste(names(fit$bestTune), fit$bestTune[1, ], sep = "=", collapse = ", "),
      "| best CV macro_auc:",
      round(max(fit$results$macro_auc, na.rm = TRUE), 4)
    )
  )
#       }
}

stopCluster(cl)
registerDoSEQ()

metric_df <- bind_rows(model_results)

print("Finish training")


## Refit the selected model using only the training partition

In [ ]:
eval_ctrl$sampling <- "smote"


cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuned",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
# Export selected tuning parameters or model results.
    best_tune <- models[[paste0(group_type,'/',m)]]$bestTune

# Analysis step: see following code.
    x_train_subset <- x_train_model[idx_train, , drop = FALSE]

# Analysis step: see following code.
    common_args_subset <- list(
      form = as.formula("Clinical.diagnosis ~ ."),
      data = x_train_subset,
      method = m,
      tuneGrid = best_tune,   # Use the selected parameter combination; do not repeat the grid search
      trControl = do.call(trainControl, eval_ctrl),
      metric = "macro_auc",
      maximize = TRUE
    )

# Analysis step: see following code.
    if (m == "xgbTree") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(eval_metric = "mlogloss", verbose = 0))
      )
    } else if (m == "multinom") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale"), trace = FALSE))
      )
    } else if (m %in% c("pda", "svmRadial")) {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale")))
      )
    } else {
      fit_subset <- do.call(train, common_args_subset)
    }

# Optional: Analysis step: see following code.
    models[[paste0(group_type,'/',m)]] <- fit_subset
}

stopCluster(cl)
registerDoSEQ()


In [ ]:
# Export selected tuning parameters or model results.
write_csv(metric_df, file.path(MODELING_TABLE_DIR, paste0('total_patient_metrics_split(smote_', ctrl$method, ').csv')))
# Optional: save the fitted classifier models.
saveRDS(models, file = file.path(MODELING_TABLE_DIR, paste0('fine_tuned_models_', ctrl$method, '.rds')))


## Evaluate the fitted model on held-out test data

In [ ]:
# test_eval <- list()

lev <- levels(y_test)

for (m in names(models)) {
  met <- str_split(m, '/')[[1]][2]
  if (str_detect(m, group_type) & (met %in% methods)){
  message("Evaluating ", m, " on held-out test set...")
  
  test_data_m <- make_test_prediction_data(
    fit = models[[m]],
    x_test_model = x_test_model,
    y_test = y_test
  )
  
  eval_m <- evaluate_with_bootstrap(
    test_data = test_data_m,
    summary_fun = multi_auc_summary,
    lev = lev,
    R = 1000,
    conf = 0.95,
    seed = 2026
  )
  
  test_predictions[[m]] <- test_data_m
#   test_eval[[m]] <- eval_m
  
  test_metrics[[m]] <- data.frame(
    model = m,
    eval_m$summary,
    row.names = NULL
  ) %>% separate(model, into = c("group", "model"), sep = "/", extra = "merge")
}
}

test_metrics_df <- do.call(rbind, test_metrics)
rownames(test_metrics_df) <- NULL

# test_metrics_df


In [ ]:
# Analysis step: see following code.
test_metrics_df$formatted <- sprintf(
  "%.3f (%.3f–%.3f)",
  test_metrics_df$estimate,
  test_metrics_df$ci_lower,
  test_metrics_df$ci_upper
)

test_metrics_wide <- reshape(
  test_metrics_df[, c("model", "group", "metric", "formatted")],
  idvar = c("model", "group"),
  timevar = "metric",
  direction = "wide"
)

names(test_metrics_wide) <- sub("^formatted\\.", "", names(test_metrics_wide))

# test_metrics_wide


In [ ]:
final_metric_df <- metric_df %>%
  group_by(model, group) %>%
  slice_max(macro_auc, n = 1, with_ties = FALSE) %>%  # Retain the best macro-AUC configuration per group
  ungroup() %>% as.data.frame()
print(final_metric_df)


In [ ]:
fine_tuned_params <- setdiff(names(final_metric_df), names(test_metrics_wide))
# Analysis step: see following code.
fine_tuned_params <- fine_tuned_params[!sapply(fine_tuned_params, function(x) any(grepl("SD", x, fixed = TRUE)))]
fine_tuned_params <- c(c('model', 'group'),fine_tuned_params)
# fine_tuned_params


In [ ]:
test_metrics_wide <- final_metric_df %>% dplyr::select(fine_tuned_params) %>% merge(test_metrics_wide, by = c('model', 'group'))
print(test_metrics_wide %>% filter(group == group_type))


In [ ]:
# Export selected tuning parameters or model results.
write_csv(test_metrics_wide, file.path(MODELING_TABLE_DIR, paste0('patient_metrics_split(smote_', ctrl$method, ').csv')))


# DT_UIH_Manual Group

## Extract planning-group training and held-out testing data

In [ ]:
group_type <- "DT_UIH_Manual"
x_train <- train_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_train <- train_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
x_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
ranger_param$mtry <- c(floor(sqrt(ncol(x_train)-2)/2), 
                                  floor(sqrt(ncol(x_train)-2)),
                                  floor((ncol(x_train)-2)/3), 10)
Grid$ranger <- expand.grid(ranger_param)


In [ ]:
ordered_factor_cols <- names(x_train)[sapply(x_train, is.ordered)]
# print(ordered_factor_cols)

x_train_ord <- x_train
x_test_ord <- x_test

# Analysis step: see following code.
for (v in ordered_factor_cols) {
  x_train_ord[[v]] <- as.numeric(x_train_ord[[v]])
  x_test_ord[[v]] <- as.numeric(x_test_ord[[v]])
}

# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_train_ord,
  fullRank = TRUE
)

x_train_model <- predict(dummy_obj, newdata = x_train_ord) |> as.data.frame()
x_train_model$Clinical.diagnosis <- y_train
# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_test_ord,
  fullRank = TRUE
)

x_test_model <- predict(dummy_obj, newdata = x_test_ord) |> as.data.frame()


In [ ]:
# Analysis step: see following code.
x_test_1 <- x_test_model %>% mutate(Clinical.diagnosis = y_test)
idx_train <- seq_len(nrow(x_train_model))
idx_valid <- (nrow(x_train_model) + 1):(nrow(x_train_model)+nrow(x_test_1))
x_train_model <- x_train_model %>% bind_rows(x_test_1)
rownames(x_train_model) <- seq_len(nrow(x_train_model))


In [ ]:
nrow(x_train_model[idx_train,])
length(idx_train)


## Tune classifier hyperparameters

In [ ]:
# Analysis step: see following code.
# ctrl <- ff_ctrl


cv_ctrl$sampling <- "up"
cv_ctrl$index <- list(Holdout = idx_train)
cv_ctrl$indexOut <- list(Holdout = idx_valid)
ctrl <- cv_ctrl  # Train using the predefined partition

# Analysis step: see following code.
cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuning",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
  
  common_args <- list(
    form = as.formula("Clinical.diagnosis ~ ."),
    data = x_train_model,
#     y = y_train,
    method = m,
    tuneGrid = Grid[[m]],
    trControl = do.call(trainControl, ctrl),
    metric = "macro_auc",
    maximize = TRUE
  )
  
  if (m == "xgbTree") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          eval_metric = "mlogloss",
          verbose = 0
        )
      )
    )
    
  } else if (m == "multinom") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale"),
          trace = FALSE
        )
      )
    )
    
  } else if (m %in% c("pda", "svmRadial")) {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale")
        )
      )
    )
    
  } else {
    
    fit <- do.call(train, common_args)
  }
  
  models[[paste0(group_type,'/',m)]] <- fit
  
  model_results[[paste0(group_type,'/',m)]] <- fit$results %>%
    mutate(
      model = m,
      group = group_type
    )
  
  message(
    paste(
      "Finished", m,
      "| selected tuning parameters:",
      paste(names(fit$bestTune), fit$bestTune[1, ], sep = "=", collapse = ", "),
      "| best CV macro_auc:",
      round(max(fit$results$macro_auc, na.rm = TRUE), 4)
    )
  )
#       }
}

stopCluster(cl)
registerDoSEQ()

metric_df <- bind_rows(model_results)

print("Finish training")


## Refit the selected model using only the training partition

In [ ]:
eval_ctrl$sampling <- "up"


cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuned",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
# Export selected tuning parameters or model results.
    best_tune <- models[[paste0(group_type,'/',m)]]$bestTune

# Analysis step: see following code.
    x_train_subset <- x_train_model[idx_train, , drop = FALSE]

# Analysis step: see following code.
    common_args_subset <- list(
      form = as.formula("Clinical.diagnosis ~ ."),
      data = x_train_subset,
      method = m,
      tuneGrid = best_tune,   # Use the selected parameter combination; do not repeat the grid search
      trControl = do.call(trainControl, eval_ctrl),
      metric = "macro_auc",
      maximize = TRUE
    )

# Analysis step: see following code.
    if (m == "xgbTree") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(eval_metric = "mlogloss", verbose = 0))
      )
    } else if (m == "multinom") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale"), trace = FALSE))
      )
    } else if (m %in% c("pda", "svmRadial")) {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale")))
      )
    } else {
      fit_subset <- do.call(train, common_args_subset)
    }

# Optional: Analysis step: see following code.
    models[[paste0(group_type,'/',m)]] <- fit_subset
}

stopCluster(cl)
registerDoSEQ()


In [ ]:
# Export selected tuning parameters or model results.
write_csv(metric_df, file.path(MODELING_TABLE_DIR, paste0('total_patient_metrics_split(smote_', ctrl$method, ').csv')))
# Optional: save the fitted classifier models.
saveRDS(models, file = file.path(MODELING_TABLE_DIR, paste0('fine_tuned_models_', ctrl$method, '.rds')))


## Evaluate the fitted model on held-out test data

In [ ]:
# test_eval <- list()

lev <- levels(y_test)

for (m in names(models)) {
  met <- str_split(m, '/')[[1]][2]
  if (str_detect(m, group_type) & (met %in% methods)){
  message("Evaluating ", m, " on held-out test set...")
  
  test_data_m <- make_test_prediction_data(
    fit = models[[m]],
    x_test_model = x_test_model,
    y_test = y_test
  )
  
  eval_m <- evaluate_with_bootstrap(
    test_data = test_data_m,
    summary_fun = multi_auc_summary,
    lev = lev,
    R = 1000,
    conf = 0.95,
    seed = 2026
  )
  
  test_predictions[[m]] <- test_data_m
#   test_eval[[m]] <- eval_m
  
  test_metrics[[m]] <- data.frame(
    model = m,
    eval_m$summary,
    row.names = NULL
  ) %>% separate(model, into = c("group", "model"), sep = "/", extra = "merge")
}
}

test_metrics_df <- do.call(rbind, test_metrics)
rownames(test_metrics_df) <- NULL

# test_metrics_df


In [ ]:
# Analysis step: see following code.
test_metrics_df$formatted <- sprintf(
  "%.3f (%.3f–%.3f)",
  test_metrics_df$estimate,
  test_metrics_df$ci_lower,
  test_metrics_df$ci_upper
)

test_metrics_wide <- reshape(
  test_metrics_df[, c("model", "group", "metric", "formatted")],
  idvar = c("model", "group"),
  timevar = "metric",
  direction = "wide"
)

names(test_metrics_wide) <- sub("^formatted\\.", "", names(test_metrics_wide))

# test_metrics_wide


In [ ]:
final_metric_df <- metric_df %>%
  group_by(model, group) %>%
  slice_max(macro_auc, n = 1, with_ties = FALSE) %>%  # Retain the best macro-AUC configuration per group
  ungroup() %>% as.data.frame()
# print(final_metric_df %>% filter(group=='Manual'))


In [ ]:
fine_tuned_params <- setdiff(names(final_metric_df), names(test_metrics_wide))
# Analysis step: see following code.
fine_tuned_params <- fine_tuned_params[!sapply(fine_tuned_params, function(x) any(grepl("SD", x, fixed = TRUE)))]
fine_tuned_params <- c(c('model', 'group'),fine_tuned_params)
# fine_tuned_params


In [ ]:
test_metrics_wide <- final_metric_df %>% dplyr::select(fine_tuned_params) %>% merge(test_metrics_wide, by = c('model', 'group'))
print(test_metrics_wide %>% filter(group==group_type))


In [ ]:
# Export selected tuning parameters or model results.
write_csv(test_metrics_wide, file.path(MODELING_TABLE_DIR, paste0('patient_metrics_split(smote_', ctrl$method, ').csv')))


# DT_UIH_AI Group

## Extract planning-group training and held-out testing data

In [ ]:
group_type <- "DT_UIH_AI"
x_train <- train_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_train <- train_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
x_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
ranger_param$mtry <- c(floor(sqrt(ncol(x_train)-2)/2), 
                                  floor(sqrt(ncol(x_train)-2)),
                                  floor((ncol(x_train)-2)/3), 10)
Grid$ranger <- expand.grid(ranger_param)


In [ ]:
ordered_factor_cols <- names(x_train)[sapply(x_train, is.ordered)]
print(ordered_factor_cols)

x_train_ord <- x_train
x_test_ord <- x_test

# Analysis step: see following code.
for (v in ordered_factor_cols) {
  x_train_ord[[v]] <- as.numeric(x_train_ord[[v]])
  x_test_ord[[v]] <- as.numeric(x_test_ord[[v]])
}

# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_train_ord,
  fullRank = TRUE
)

x_train_model <- predict(dummy_obj, newdata = x_train_ord) |> as.data.frame()
x_train_model$Clinical.diagnosis <- y_train
# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_test_ord,
  fullRank = TRUE
)

x_test_model <- predict(dummy_obj, newdata = x_test_ord) |> as.data.frame()

# print(str(x_train_model))


In [ ]:
# Analysis step: see following code.
x_test_1 <- x_test_model %>% mutate(Clinical.diagnosis = y_test)
idx_train <- seq_len(nrow(x_train_model))
idx_valid <- (nrow(x_train_model) + 1):(nrow(x_train_model)+nrow(x_test_1))
x_train_model <- x_train_model %>% bind_rows(x_test_1)
rownames(x_train_model) <- seq_len(nrow(x_train_model))


## Tune classifier hyperparameters

In [ ]:
# Analysis step: see following code.
# ctrl <- ff_ctrl


cv_ctrl$sampling <- "up"
cv_ctrl$index <- list(Holdout = idx_train)
cv_ctrl$indexOut <- list(Holdout = idx_valid)
ctrl <- cv_ctrl  # Train using the predefined partition

# Analysis step: see following code.
cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuning",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
  
  common_args <- list(
    form = as.formula("Clinical.diagnosis ~ ."),
    data = x_train_model,
#     y = y_train,
    method = m,
    tuneGrid = Grid[[m]],
    trControl = do.call(trainControl, ctrl),
    metric = "macro_auc",
    maximize = TRUE
  )
  
  if (m == "xgbTree") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          eval_metric = "mlogloss",
          verbose = 0
        )
      )
    )
    
  } else if (m == "multinom") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale"),
          trace = FALSE
        )
      )
    )
    
  } else if (m %in% c("pda", "svmRadial")) {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale")
        )
      )
    )
    
  } else {
    
    fit <- do.call(train, common_args)
  }
  
  models[[paste0(group_type,'/',m)]] <- fit
  
  model_results[[paste0(group_type,'/',m)]] <- fit$results %>%
    mutate(
      model = m,
      group = group_type
    )
  
  message(
    paste(
      "Finished", m,
      "| selected tuning parameters:",
      paste(names(fit$bestTune), fit$bestTune[1, ], sep = "=", collapse = ", "),
      "| best CV macro_auc:",
      round(max(fit$results$macro_auc, na.rm = TRUE), 4)
    )
  )
#       }
}

stopCluster(cl)
registerDoSEQ()

metric_df <- bind_rows(model_results)

print("Finish training")


## Refit the selected model using only the training partition

In [ ]:
eval_ctrl$sampling <- "up"


cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuned",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
# Export selected tuning parameters or model results.
    best_tune <- models[[paste0(group_type,'/',m)]]$bestTune

# Analysis step: see following code.
    x_train_subset <- x_train_model[idx_train, , drop = FALSE]

# Analysis step: see following code.
    common_args_subset <- list(
      form = as.formula("Clinical.diagnosis ~ ."),
      data = x_train_subset,
      method = m,
      tuneGrid = best_tune,   # Use the selected parameter combination; do not repeat the grid search
      trControl = do.call(trainControl, eval_ctrl),
      metric = "macro_auc",
      maximize = TRUE
    )

# Analysis step: see following code.
    if (m == "xgbTree") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(eval_metric = "mlogloss", verbose = 0))
      )
    } else if (m == "multinom") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale"), trace = FALSE))
      )
    } else if (m %in% c("pda", "svmRadial")) {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale")))
      )
    } else {
      fit_subset <- do.call(train, common_args_subset)
    }

# Optional: Analysis step: see following code.
    models[[paste0(group_type,'/',m)]] <- fit_subset
}

stopCluster(cl)
registerDoSEQ()


In [ ]:
# Export selected tuning parameters or model results.
write_csv(metric_df, file.path(MODELING_TABLE_DIR, paste0('total_patient_metrics_split(smote_', ctrl$method, ').csv')))
# Optional: save the fitted classifier models.
saveRDS(models, file = file.path(MODELING_TABLE_DIR, paste0('fine_tuned_models_', ctrl$method, '.rds')))


## Evaluate the fitted model on held-out test data

In [ ]:
# test_eval <- list()

lev <- levels(y_test)

for (m in names(models)) {
  met <- str_split(m, '/')[[1]][2]
  if (str_detect(m, group_type) & (met %in% methods)){
  message("Evaluating ", m, " on held-out test set...")
  
  test_data_m <- make_test_prediction_data(
    fit = models[[m]],
    x_test_model = x_test_model,
    y_test = y_test
  )
  
  eval_m <- evaluate_with_bootstrap(
    test_data = test_data_m,
    summary_fun = multi_auc_summary,
    lev = lev,
    R = 1000,
    conf = 0.95,
    seed = 2026
  )
  
  test_predictions[[m]] <- test_data_m
#   test_eval[[m]] <- eval_m
  
  test_metrics[[m]] <- data.frame(
    model = m,
    eval_m$summary,
    row.names = NULL
  ) %>% separate(model, into = c("group", "model"), sep = "/", extra = "merge")
}
}

test_metrics_df <- do.call(rbind, test_metrics)
rownames(test_metrics_df) <- NULL

# test_metrics_df


In [ ]:
# Analysis step: see following code.
test_metrics_df$formatted <- sprintf(
  "%.3f (%.3f–%.3f)",
  test_metrics_df$estimate,
  test_metrics_df$ci_lower,
  test_metrics_df$ci_upper
)

test_metrics_wide <- reshape(
  test_metrics_df[, c("model", "group", "metric", "formatted")],
  idvar = c("model", "group"),
  timevar = "metric",
  direction = "wide"
)

names(test_metrics_wide) <- sub("^formatted\\.", "", names(test_metrics_wide))

# test_metrics_wide


In [ ]:
final_metric_df <- metric_df %>%
  group_by(model, group) %>%
  slice_max(macro_auc, n = 1, with_ties = FALSE) %>%  # Retain the best macro-AUC configuration per group
  ungroup() %>% as.data.frame()
# print(final_metric_df)


In [ ]:
fine_tuned_params <- setdiff(names(final_metric_df), names(test_metrics_wide))
# Analysis step: see following code.
fine_tuned_params <- fine_tuned_params[!sapply(fine_tuned_params, function(x) any(grepl("SD", x, fixed = TRUE)))]
fine_tuned_params <- c(c('model', 'group'),fine_tuned_params)
fine_tuned_params


In [ ]:
test_metrics_wide <- final_metric_df %>% dplyr::select(fine_tuned_params) %>% merge(test_metrics_wide, by = c('model', 'group'))
print(test_metrics_wide %>% filter(group==group_type))


In [ ]:
# Export selected tuning parameters or model results.
write_csv(test_metrics_wide, file.path(MODELING_TABLE_DIR, paste0('patient_metrics_split(smote_', ctrl$method, ').csv')))


# DT_UIH_Interactive Group

## Extract planning-group training and held-out testing data

In [ ]:
group_type <- "DT_UIH_Interactive"
x_train <- train_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_train <- train_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
x_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% dplyr::select(features)
y_test <- test_df %>% filter(str_detect(ScanGroup, group_type)) %>% pull(Clinical.diagnosis)
ranger_param$mtry <- c(floor(sqrt(ncol(x_train)-2)/2), 
                                  floor(sqrt(ncol(x_train)-2)),
                                  floor((ncol(x_train)-2)/3), 10)
Grid$ranger <- expand.grid(ranger_param)


In [ ]:
ordered_factor_cols <- names(x_train)[sapply(x_train, is.ordered)]
print(ordered_factor_cols)

x_train_ord <- x_train
x_test_ord <- x_test

# Analysis step: see following code.
for (v in ordered_factor_cols) {
  x_train_ord[[v]] <- as.numeric(x_train_ord[[v]])
  x_test_ord[[v]] <- as.numeric(x_test_ord[[v]])
}

# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_train_ord,
  fullRank = TRUE
)

x_train_model <- predict(dummy_obj, newdata = x_train_ord) |> as.data.frame()
x_train_model$Clinical.diagnosis <- y_train
# Analysis step: see following code.
dummy_obj <- dummyVars(
  ~ .,
  data = x_test_ord,
  fullRank = TRUE
)

x_test_model <- predict(dummy_obj, newdata = x_test_ord) |> as.data.frame()

# print(str(x_train_model))


In [ ]:
# Analysis step: see following code.
x_test_1 <- x_test_model %>% mutate(Clinical.diagnosis = y_test)
idx_train <- seq_len(nrow(x_train_model))
idx_valid <- (nrow(x_train_model) + 1):(nrow(x_train_model)+nrow(x_test_1))
x_train_model <- x_train_model %>% bind_rows(x_test_1)
rownames(x_train_model) <- seq_len(nrow(x_train_model))


## Tune classifier hyperparameters

In [ ]:
# Analysis step: see following code.
# ctrl <- ff_ctrl


cv_ctrl$sampling <- "up"
cv_ctrl$index <- list(Holdout = idx_train)
cv_ctrl$indexOut <- list(Holdout = idx_valid)
ctrl <- cv_ctrl  # Train using the predefined partition

# Analysis step: see following code.
cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuning",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
  
  common_args <- list(
    form = as.formula("Clinical.diagnosis ~ ."),
    data = x_train_model,
#     y = y_train,
    method = m,
    tuneGrid = Grid[[m]],
    trControl = do.call(trainControl, ctrl),
    metric = "macro_auc",
    maximize = TRUE
  )
  
  if (m == "xgbTree") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          eval_metric = "mlogloss",
          verbose = 0
        )
      )
    )
    
  } else if (m == "multinom") {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale"),
          trace = FALSE
        )
      )
    )
    
  } else if (m %in% c("pda", "svmRadial")) {
    
    fit <- do.call(
      train,
      c(
        common_args,
        list(
          preProcess = c("center", "scale")
        )
      )
    )
    
  } else {
    
    fit <- do.call(train, common_args)
  }
  
  models[[paste0(group_type,'/',m)]] <- fit
  
  model_results[[paste0(group_type,'/',m)]] <- fit$results %>%
    mutate(
      model = m,
      group = group_type
    )
  
  message(
    paste(
      "Finished", m,
      "| selected tuning parameters:",
      paste(names(fit$bestTune), fit$bestTune[1, ], sep = "=", collapse = ", "),
      "| best CV macro_auc:",
      round(max(fit$results$macro_auc, na.rm = TRUE), 4)
    )
  )
#       }
}

stopCluster(cl)
registerDoSEQ()

metric_df <- bind_rows(model_results)

print("Finish training")


## Refit the selected model using only the training partition

In [ ]:
eval_ctrl$sampling <- "up"


cl <- makePSOCKcluster(12)
clusterEvalQ(cl, {
  library(caret)
  library(pROC)
})
registerDoParallel(cl)
# registerDoSEQ()

for (m in methods) {
#   if (m=='svmRadial'){
  message(
    paste(
      "Training", m,
      "by fine tuned",
      paste0(names(Grid[[m]]), collapse = ", "),
      "..."
    )
  )
# Export selected tuning parameters or model results.
    best_tune <- models[[paste0(group_type,'/',m)]]$bestTune

# Analysis step: see following code.
    x_train_subset <- x_train_model[idx_train, , drop = FALSE]

# Analysis step: see following code.
    common_args_subset <- list(
      form = as.formula("Clinical.diagnosis ~ ."),
      data = x_train_subset,
      method = m,
      tuneGrid = best_tune,   # Use the selected parameter combination; do not repeat the grid search
      trControl = do.call(trainControl, eval_ctrl),
      metric = "macro_auc",
      maximize = TRUE
    )

# Analysis step: see following code.
    if (m == "xgbTree") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(eval_metric = "mlogloss", verbose = 0))
      )
    } else if (m == "multinom") {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale"), trace = FALSE))
      )
    } else if (m %in% c("pda", "svmRadial")) {
      fit_subset <- do.call(
        train,
        c(common_args_subset,
          list(preProcess = c("center", "scale")))
      )
    } else {
      fit_subset <- do.call(train, common_args_subset)
    }

# Optional: Analysis step: see following code.
    models[[paste0(group_type,'/',m)]] <- fit_subset
}

stopCluster(cl)
registerDoSEQ()


In [ ]:
# Export selected tuning parameters or model results.
write_csv(metric_df, file.path(MODELING_TABLE_DIR, paste0('total_patient_metrics_split(smote_', ctrl$method, ').csv')))
# Optional: save the fitted classifier models.
saveRDS(models, file = file.path(MODELING_TABLE_DIR, paste0('fine_tuned_models_', ctrl$method, '.rds')))


## Evaluate the fitted model on held-out test data

In [ ]:
# test_eval <- list()

lev <- levels(y_test)

for (m in names(models)) {
  met <- str_split(m, '/')[[1]][2]
  if (str_detect(m, group_type) & (met %in% methods)){
  message("Evaluating ", m, " on held-out test set...")
  
  test_data_m <- make_test_prediction_data(
    fit = models[[m]],
    x_test_model = x_test_model,
    y_test = y_test
  )
  
  eval_m <- evaluate_with_bootstrap(
    test_data = test_data_m,
    summary_fun = multi_auc_summary,
    lev = lev,
    R = 1000,
    conf = 0.95,
    seed = 2026
  )
  
  test_predictions[[m]] <- test_data_m
#   test_eval[[m]] <- eval_m
  
  test_metrics[[m]] <- data.frame(
    model = m,
    eval_m$summary,
    row.names = NULL
  ) %>% separate(model, into = c("group", "model"), sep = "/", extra = "merge")
}
}

test_metrics_df <- do.call(rbind, test_metrics)
rownames(test_metrics_df) <- NULL

# test_metrics_df


In [ ]:
# Analysis step: see following code.
test_metrics_df$formatted <- sprintf(
  "%.3f (%.3f–%.3f)",
  test_metrics_df$estimate,
  test_metrics_df$ci_lower,
  test_metrics_df$ci_upper
)

test_metrics_wide <- reshape(
  test_metrics_df[, c("model", "group", "metric", "formatted")],
  idvar = c("model", "group"),
  timevar = "metric",
  direction = "wide"
)

names(test_metrics_wide) <- sub("^formatted\\.", "", names(test_metrics_wide))

# test_metrics_wide


In [ ]:
final_metric_df <- metric_df %>%
  group_by(model, group) %>%
  slice_max(macro_auc, n = 1, with_ties = FALSE) %>%  # Retain the best macro-AUC configuration per group
  ungroup() %>% as.data.frame()
print(final_metric_df)


In [ ]:
fine_tuned_params <- setdiff(names(final_metric_df), names(test_metrics_wide))
# Analysis step: see following code.
fine_tuned_params <- fine_tuned_params[!sapply(fine_tuned_params, function(x) any(grepl("SD", x, fixed = TRUE)))]
fine_tuned_params <- c(c('model', 'group'),fine_tuned_params)
fine_tuned_params


In [ ]:
test_metrics_wide <- final_metric_df %>% dplyr::select(fine_tuned_params) %>% merge(test_metrics_wide, by = c('model', 'group'))
print(test_metrics_wide %>% filter(group==group_type))


In [ ]:
# Export selected tuning parameters or model results.
write_csv(test_metrics_wide, file.path(MODELING_TABLE_DIR, paste0('patient_metrics_split(smote_', ctrl$method, ').csv')))


In [ ]:
print(test_metrics_wide %>% filter(model %in% c('glmnet', 'kknn')))
